<a href="https://colab.research.google.com/github/akshita-singh-2808/Accenture_DSA/blob/main/Accenture_SQL_Practice_With_Tables.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Accenture SQL Practice — Questions, Tables & Auto-Check

**Based on your uploaded `Accenture SQL.pdf` (30 questions).** Use this notebook in Google Colab to practise SQL without repeatedly inspecting the schema.

For every question, you will see:
- the relevant table contents (as readable tables)
- each table's columns and data types
- a **JOIN guide** showing the relationship between tables, where applicable
- a blank SQL answer cell and an automatic result checker

The data is a small practice dataset built for this notebook, not the original Accenture database. Run the setup and data cells once, then for each question run its **Table Preview** cell before writing your query.

## 1. Setup — run this cell first

In [65]:
import sqlite3
import pandas as pd
import re
from IPython.display import display

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")

# Compatibility helpers for common SQL expressions used in interview platforms.
conn.create_function("REGEXP", 2, lambda pattern, value: bool(re.search(pattern, str(value or ""))))
conn.create_function("CONCAT", -1, lambda *args: "".join("" if x is None else str(x) for x in args))

def run_sql(query):
    """Run a SQL query and display the results as a DataFrame."""
    if not query or not query.strip():
        print("Write your SQL query first.")
        return None
    return display(pd.read_sql_query(query, conn))

print("Setup ready. Write SQL in the answer cells below.")

def show_question_tables(question_number, table_names, join_guide=None):
    """Show all relevant sample tables, their columns/types, and suggested join relationships."""
    print(f"QUESTION {question_number} — TABLE PREVIEW")
    if join_guide:
        print("\nJOIN GUIDE (relationships only; write the SQL yourself):")
        for relationship in join_guide:
            print("  • " + relationship)
    else:
        print("\nNo JOIN needed for this question.")

    for table_name in table_names:
        print(f"\n{'=' * 14} TABLE: {table_name} {'=' * 14}")
        schema = pd.read_sql_query(f"PRAGMA table_info({table_name})", conn)
        if not schema.empty:
            schema_view = schema[['name', 'type', 'pk']].copy()
            schema_view.columns = ['column', 'data type', 'primary key?']
            display(schema_view)
        data = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
        print(f"Rows: {len(data)}")
        display(data)


Setup ready. Write SQL in the answer cells below.


## 2. Create sample tables and data — run this cell once

In [66]:
conn.executescript("""
-- Q1: Transactions
CREATE TABLE transactions (
    transaction_id INTEGER PRIMARY KEY,
    amount REAL,
    transaction_type TEXT
);
INSERT INTO transactions VALUES
(1001, 15000, 'Debit'), (1002, 52000, 'Debit'), (1003, 24000, 'Credit'),
(1004, 42000, 'Debit'), (1005, 9000, 'Debit');

-- Q2, Q12, Q13: Customers and accounts
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT,
    contact TEXT
);
CREATE TABLE accounts (
    account_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    account_type_id INTEGER,
    account_type TEXT,
    account_balance REAL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
INSERT INTO customers VALUES
(1, 'Aarav', 'Shah', '9000000001'), (2, 'Meera', 'Rao', '9000000002'),
(3, 'Kabir', 'Das', '9000000003'), (4, 'Isha', 'Patel', '9000000004'),
(5, 'Rohan', 'Singh', '9000000005');
INSERT INTO accounts VALUES
(501, 1, 10, 'Savings', 75000), (502, 2, 10, 'Savings Plus', 45000),
(503, 3, 20, 'Current', 125000), (504, 4, 30, 'Salary', 68000),
(505, 5, 20, 'Current', 25000), (506, 1, 20, 'Current', 55000);

-- Q3: Employee payroll
CREATE TABLE employee_payroll (
    empid INTEGER PRIMARY KEY,
    empname TEXT,
    basic REAL,
    netpay REAL
);
INSERT INTO employee_payroll VALUES
(11, 'Ananya Verma', 42000, 51000), (12, 'Dev Kumar', 55000, 67000),
(13, 'Sara Khan', 38500, 46000), (14, 'Arjun Mehta', 62000, 74000);

-- Q4, Q10, Q11: Employee details, departments, salaries
CREATE TABLE employee_info (
    empid INTEGER PRIMARY KEY,
    empname TEXT,
    experience_years REAL,
    joining_date TEXT,
    deptid INTEGER,
    employee_category TEXT
);
CREATE TABLE department_info (
    deptid INTEGER PRIMARY KEY,
    deptname TEXT,
    location TEXT
);
CREATE TABLE salary_info (
    employee_category TEXT PRIMARY KEY,
    basic REAL,
    special_incentive_allowance REAL
);
INSERT INTO employee_info VALUES
(201, 'Ananya Verma', 7, '2018-06-15', 1, 'A'),
(202, 'Dev Kumar', 4, '2022-07-01', 2, 'B'),
(203, 'Sara Khan', 9, '2015-03-20', 3, 'A'),
(204, 'Arjun Mehta', 6, '2019-11-10', 2, 'C'),
(205, 'Meera Rao', 2, '2024-01-15', 1, 'B');
INSERT INTO department_info VALUES
(1, 'IT', 'BANGALORE'), (2, 'Analytics', 'MUMBAI'), (3, 'HR', 'BANGALORE');
INSERT INTO salary_info VALUES
('A', 85000, 12000), ('B', 62000, 8000), ('C', 95000, 15000);

-- Q5: Course schedule
CREATE TABLE courses (
    course_id INTEGER PRIMARY KEY,
    course_name TEXT,
    course_no TEXT
);
CREATE TABLE course_schedule (
    schedule_id INTEGER PRIMARY KEY,
    course_id INTEGER,
    day TEXT,
    start_time TEXT,
    faculty_name TEXT,
    FOREIGN KEY (course_id) REFERENCES courses(course_id)
);
INSERT INTO courses VALUES
(1, 'Database Systems', 'CS201'), (2, 'Data Analytics', 'CS305'),
(3, 'Thermodynamics', 'ME202'), (4, 'Business Statistics', 'MG210');
INSERT INTO course_schedule VALUES
(1, 1, 'Monday', '09:00', 'Dr Rao'), (2, 2, 'Tuesday', '11:00', 'Dr Sen'),
(3, 3, 'Wednesday', '10:00', 'Dr Rao'), (4, 4, 'Friday', '14:00', 'Dr Iyer');

-- Q6: Books
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY,
    title TEXT,
    price REAL,
    isbn TEXT,
    publish_date TEXT,
    category TEXT
);
INSERT INTO books VALUES
(1, 'Classic Tales', 450, '9780000000011', '1995-05-12', 'Classic'),
(2, 'Old Poems', 250, '9780000000028', '1975-02-01', 'Classic'),
(3, 'Modern SQL', 700, '9780000000035', '2021-09-20', 'Technology'),
(4, 'The Odyssey', 525, '9780000000042', '2000-01-01', 'Classic');

-- Q7: Channel categories
CREATE TABLE channelcategory (
    categoryid INTEGER PRIMARY KEY,
    categoryname TEXT
);
INSERT INTO channelcategory VALUES
(1, 'Movies'), (2, 'Music'), (3, 'Sports'), (4, 'News'), (5, 'Family');

-- Q8, Q22: Trains and stations
CREATE TABLE train_station_tbl (
    station_id INTEGER PRIMARY KEY,
    station_name TEXT
);
CREATE TABLE train_details_tbl (
    train_id INTEGER PRIMARY KEY,
    train_name TEXT,
    train_type TEXT,
    train_to INTEGER,
    train_speed REAL,
    FOREIGN KEY (train_to) REFERENCES train_station_tbl(station_id)
);
INSERT INTO train_station_tbl VALUES
(1, 'PUNE'), (2, 'MUMBAI'), (3, 'DELHI'), (4, 'CHENNAI');
INSERT INTO train_details_tbl VALUES
(101, 'Mumbai Express', 'Express', 1, 75), (102, 'Pune Mail', 'Mail', 1, 45),
(103, 'Metro Local', 'Passenger', 2, 35), (104, 'Delhi Superfast', 'Superfast', 3, 110),
(105, 'Morning Intercity', 'Intercity', 4, 48);

-- Q9: Leave information
CREATE TABLE leave_info (
    empid INTEGER,
    leave_type TEXT,
    total_leaves INTEGER
);
INSERT INTO leave_info VALUES
(11, 'CL', 12), (12, 'ML', 14), (13, 'EL', 18), (14, 'CL', 8), (15, 'ML', 22);

-- Q12 uses accounts above

-- Q14: Staff
CREATE TABLE staff (
    staff_id INTEGER PRIMARY KEY,
    firstname TEXT,
    position TEXT,
    salary REAL
);
INSERT INTO staff VALUES
(1, 'Nisha', 'Analyst', 95000), (2, 'Vikram', 'Manager', 125000),
(3, 'Fatima', 'Associate', 72000), (4, 'Karan', 'Consultant', 102000);

-- Q15: Patient billing
CREATE TABLE patient (
    patient_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT,
    email TEXT,
    admissiondate TEXT
);
CREATE TABLE billing (
    billing_id INTEGER PRIMARY KEY,
    patient_id INTEGER,
    totalamount REAL,
    paymentstatus TEXT,
    FOREIGN KEY (patient_id) REFERENCES patient(patient_id)
);
INSERT INTO patient VALUES
(1, 'Aditi', 'Sharma', 'aditi@example.com', '2024-01-10'),
(2, 'Rahul', 'Jain', 'rahul@example.com', '2024-02-05'),
(3, 'Neha', 'Kapoor', 'neha@example.com', '2024-03-11');
INSERT INTO billing VALUES
(101, 1, 25000, 'Unpaid'), (102, 1, 7000, 'Unpaid'),
(103, 2, 18000, 'Paid'), (104, 3, 46000, 'Unpaid');

-- Q16, Q17, Q19, Q20, Q23: Airline and flight data
CREATE TABLE airline (
    airline_id INTEGER PRIMARY KEY,
    name TEXT,
    country TEXT
);
CREATE TABLE airplane (
    airplane_id INTEGER PRIMARY KEY,
    airline_id INTEGER,
    modelnumber TEXT,
    manufacturer TEXT,
    FOREIGN KEY (airline_id) REFERENCES airline(airline_id)
);
CREATE TABLE flight (
    flight_id TEXT PRIMARY KEY,
    airplane_id INTEGER,
    departure_date TEXT,
    departure_time TEXT,
    arrival_date TEXT,
    flight_from TEXT,
    flight_to TEXT,
    FOREIGN KEY (airplane_id) REFERENCES airplane(airplane_id)
);
CREATE TABLE cabincrew (
    cabincrew_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT,
    contact TEXT,
    flight_id TEXT,
    FOREIGN KEY (flight_id) REFERENCES flight(flight_id)
);
CREATE TABLE passenger (
    passenger_id INTEGER PRIMARY KEY,
    first_name TEXT,
    contact TEXT
);
CREATE TABLE boardingpass (
    boardingpass_id INTEGER PRIMARY KEY,
    passenger_id INTEGER,
    flight_id TEXT,
    baggage REAL,
    meal TEXT,
    FOREIGN KEY (passenger_id) REFERENCES passenger(passenger_id),
    FOREIGN KEY (flight_id) REFERENCES flight(flight_id)
);
INSERT INTO airline VALUES
(1, 'Singapore Airlines', 'Singapore'), (2, 'Air India', 'India'), (3, 'Lufthansa', 'Germany');
INSERT INTO airplane VALUES
(1, 1, 'A350-900', 'Airbus'), (2, 2, 'B787-8', 'Boeing'),
(3, 3, 'A320neo', 'Airbus'), (4, 2, 'A321', 'Airbus');
INSERT INTO flight VALUES
('F101', 1, '2024-02-10', '08:30', '2024-02-11', 'Hong Kong', 'Paris'),
('F211', 2, '2024-02-10', '11:15', '2024-02-11', 'Hong Kong', 'Paris'),
('F304', 3, '2024-02-11', '14:00', '2024-02-11', 'Delhi', 'Frankfurt'),
('F411', 4, '2024-02-11', '19:45', '2024-02-11', 'Hong Kong', 'Singapore'),
('4', 2, '2024-02-11', '21:00', '2024-02-12', 'Hong Kong', 'Singapore');
INSERT INTO cabincrew VALUES
(1, 'Anika', 'Sen', '8000000001', 'F101'), (2, 'Aman', 'Rao', '8000000002', 'F211'),
(3, 'Rita', 'Das', '8000000003', 'F304'), (4, 'Arun', 'Jain', '8000000004', 'F411');
INSERT INTO passenger VALUES
(1, 'Maya', '7000000001'), (2, 'Rohan', '7000000002'),
(3, 'Ira', '7000000003'), (4, 'Kunal', '7000000004');
INSERT INTO boardingpass VALUES
(1, 1, 'F101', 22, 'Vegetarian'), (2, 2, 'F101', 18, 'Non-Vegetarian'),
(3, 3, 'F211', 25, 'Vegetarian'), (4, 4, '4', 30, 'Vegetarian');

-- Q18: Students and registrations
CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT
);
CREATE TABLE registration (
    registration_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    reg_date TEXT,
    FOREIGN KEY (student_id) REFERENCES student(student_id)
);
INSERT INTO student VALUES
(1, 'Asha', 'Menon'), (2, 'Vivek', 'Nair'), (3, 'Riya', 'Bose'), (4, 'Omar', 'Ali');
INSERT INTO registration VALUES
(1, 1, '2012-03-14'), (2, 2, '2014-08-10'),
(3, 3, '2012-11-02'), (4, 4, '2011-01-20');

-- Q21, Q24: Products, categories, and deliveries
CREATE TABLE product (
    product_id INTEGER PRIMARY KEY,
    name TEXT
);
CREATE TABLE category (
    category_id INTEGER PRIMARY KEY,
    name TEXT
);
CREATE TABLE product_category (
    product_id INTEGER,
    category_id INTEGER,
    FOREIGN KEY (product_id) REFERENCES product(product_id),
    FOREIGN KEY (category_id) REFERENCES category(category_id)
);
CREATE TABLE order_delivery (
    order_delivery_id INTEGER PRIMARY KEY,
    status TEXT
);
CREATE TABLE order_item (
    order_item_id INTEGER PRIMARY KEY,
    product_id INTEGER,
    order_delivery_id INTEGER,
    FOREIGN KEY (product_id) REFERENCES product(product_id),
    FOREIGN KEY (order_delivery_id) REFERENCES order_delivery(order_delivery_id)
);
INSERT INTO product VALUES
(1, 'Blue Dress'), (2, 'Running Shoes'), (3, 'Desk Lamp'), (4, 'Travel Bag');
INSERT INTO category VALUES
(1, 'Women'), (2, 'Sports'), (3, 'Home'), (4, 'Travel');
INSERT INTO product_category VALUES (1, 1), (2, 2), (3, 3), (4, 4), (2, 1);
INSERT INTO order_delivery VALUES
(501, 'In the transit hub'), (502, 'Delivered'), (503, 'Processing');
INSERT INTO order_item VALUES
(1, 1, 501), (2, 2, 502), (3, 3, 501), (4, 4, 503);

-- Q25: Artists
CREATE TABLE artist (
    artist_id INTEGER PRIMARY KEY,
    name TEXT
);
INSERT INTO artist VALUES
(1, 'Studio 54'), (2, 'Adele'), (3, 'Band 2'), (4, 'The Weeknd'), (5, 'Artist 007');

-- Q26: Messages
CREATE TABLE message (
    message_id INTEGER PRIMARY KEY,
    content TEXT
);
INSERT INTO message VALUES
(1, 'Hello everyone'), (2, 'Good morning'), (3, 'Say hello to the team'), (4, 'Thanks for joining');

-- Q27, Q28: Drivers, vehicles and bookings
CREATE TABLE driver (
    driver_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT,
    license_number TEXT,
    rating REAL
);
CREATE TABLE vehicle (
    vehicle_id INTEGER PRIMARY KEY,
    driver_id INTEGER,
    plate_number TEXT,
    status TEXT,
    FOREIGN KEY (driver_id) REFERENCES driver(driver_id)
);
CREATE TABLE booking (
    booking_id INTEGER PRIMARY KEY,
    vehicle_id INTEGER,
    status TEXT,
    FOREIGN KEY (vehicle_id) REFERENCES vehicle(vehicle_id)
);
INSERT INTO driver VALUES
(1, 'Aman', 'Gupta', 'DL1001', 4.8), (2, 'Nisha', 'Roy', 'DL1002', 4.3),
(3, 'Karan', 'Das', 'DL1003', 4.6), (4, 'Meera', 'Shah', 'DL1004', 4.9);
INSERT INTO vehicle VALUES
(1, 1, 'UP32AB120', 'In Use'), (2, 2, 'UP32CD321', 'In Use'),
(3, 3, 'UP32EF450', 'In Use'), (4, 4, 'UP32GH770', 'Available');
INSERT INTO booking VALUES
(701, 1, 'Confirmed'), (702, 1, 'Cancelled'), (703, 2, 'Confirmed'),
(704, 3, 'Completed'), (705, 4, 'Confirmed');

-- Q29: Viewers
CREATE TABLE viewer (
    viewer_id INTEGER PRIMARY KEY,
    viewername TEXT
);
INSERT INTO viewer VALUES
(1, 'Aarav'), (2, 'Meera'), (3, 'Aarav'), (4, 'Riya'), (5, 'Meera'), (6, 'Aarav');

-- Q30: Users, contacts and jobs
CREATE TABLE users (
    user_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT
);
CREATE TABLE contacts (
    user_id INTEGER,
    contact_id INTEGER,
    FOREIGN KEY (user_id) REFERENCES users(user_id),
    FOREIGN KEY (contact_id) REFERENCES users(user_id)
);
CREATE TABLE jobs (
    job_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    job_title TEXT,
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
INSERT INTO users VALUES
(1, 'Anaya', 'Sharma'), (2, 'Vikram', 'Singh'), (3, 'Kavya', 'Rao'),
(4, 'Arjun', 'Mehta'), (5, 'Neel', 'Kapoor');
INSERT INTO contacts VALUES (1, 2), (1, 3), (4, 5), (2, 3);
INSERT INTO jobs VALUES
(1, 2, 'Software Engineer'), (2, 3, 'Data Analyst'),
(3, 5, 'Mechanical Engineer'), (4, 4, 'Product Manager');
""")

print("Sample tables and records created. To inspect any table, try: run_sql('SELECT * FROM transactions')")

Sample tables and records created. To inspect any table, try: run_sql('SELECT * FROM transactions')


In [67]:
# Self-check: compares your query's result with the expected result for this practice dataset.
# It checks column names/order and values. For most questions, row order is ignored unless the question asks for sorting.
import hashlib, json, re

def _norm_col(c):
    return re.sub(r"[^a-z0-9]", "", str(c).lower())

def _norm_val(v):
    if pd.isna(v):
        return None
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        fv = float(v)
        if abs(fv - round(fv)) < 1e-9:
            return int(round(fv))
        return round(fv, 8)
    return str(v)

def _signature(df, order_sensitive=False):
    cols = [_norm_col(c) for c in df.columns]
    rows = [[_norm_val(v) for v in row] for row in df.itertuples(index=False, name=None)]
    if not order_sensitive:
        rows.sort(key=lambda r: json.dumps(r, sort_keys=True, ensure_ascii=False))
    payload = json.dumps({"cols": cols, "rows": rows}, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode()).hexdigest(), len(rows)

EXPECTED_SQL_SIGNATURES = {1: {'sha256': 'c886458cd4e0e64dc7a776bc204e1748ccc7ca34e9c958589e8f6f65ccde04e5', 'rows': 2}, 2: {'sha256': '996a3897a4bca4b9f3a4151490ac3018e6f6e1a02b9dc1e00a5a32bf9d43645d', 'rows': 3}, 3: {'sha256': '0184e4cae3b1cf5f352649cb8e293f146e9cd92190b4b83be076e2a83c3f15ae', 'rows': 2}, 4: {'sha256': '4fa45101f01563170c3a6d8dae0f492bb07793ff22331801b83cb808b292b9af', 'rows': 3}, 5: {'sha256': 'f2641de1e3239741848310e70c05c577ff9a2ffda9c2ef76f2c94a17f88908a9', 'rows': 4}, 6: {'sha256': 'a2ea9a71806a6785b83183cf38d438261f521d03082b2fc73c2a30b4fc35e21f', 'rows': 2}, 7: {'sha256': '908ccbeb3e419c5c0002862ab19ec8d612da5604bab70ee78837817f252cb9ec', 'rows': 3}, 8: {'sha256': '49a29d603fb9c84ee22423c20bfc142f749289c5ddb36e121cad25c6da2d4cef', 'rows': 2}, 9: {'sha256': 'b1ee3de0c47580f9f10e0415e561d6043630bdc37180d81844b47e22d924af20', 'rows': 3}, 10: {'sha256': '3bd8a8a240f21b9fb3628c55cfae09b646d66680ecdd22ee1daf5a29bdf99f19', 'rows': 2}, 11: {'sha256': '64276a574e963c468e2c0d1c199e4c23eccdb15a964483b5d0c3bb05781dbba9', 'rows': 3}, 12: {'sha256': 'e5b14411948dfed02b0f19af9661f834334e504527dd3920e9988ca2cf2f3c22', 'rows': 3}, 13: {'sha256': '417ad31431920ef1b34a0e8f5aefb7b8bb0928574867a033bd6d076b2ad68d15', 'rows': 4}, 14: {'sha256': '7f9ca61900070bfe08038401c4b91db6dc34471272996628d37c1c7afc828d65', 'rows': 3}, 15: {'sha256': '6e85c9bcb1f0b8e15616f3651eb634b8c7f00d29db1ec7be5f028e7ae5937b48', 'rows': 2}, 16: {'sha256': '5b6fff72a1ed21a413a55a487bd10bd908524d2027a0097a39a4a8a03fc57565', 'rows': 1}, 17: {'sha256': '1dfa2723518f10ae85edc92ff7bdabd48aab83a7b38d3673881b2e825cfbde4d', 'rows': 3}, 18: {'sha256': 'ed2f512d9864d547a4091543ca083b21a804dfe7f13f09f219b432b383ecd8cc', 'rows': 2}, 19: {'sha256': '673a35de7e3e605aedf09e3c148a3f20017703e184bbdb62dfb0b2e4de07e673', 'rows': 3}, 20: {'sha256': '6fee9d3b4ec9615bb7b30270f3d7d3b161e911cb99faf01521da49ee6f320f7e', 'rows': 2}, 21: {'sha256': '1828d290a145b808df4aa0197c48f5f081581f2111ee0a0ec26a46a39f5ca2cf', 'rows': 1}, 22: {'sha256': '9846c5e87ca166f7a1144ca6432a6689c42c90b4ad340caea5276f2f9d88cf4e', 'rows': 3}, 23: {'sha256': 'e501717b5620bcae20e629bcce2b34ea54f8cd30000d020dd97ecb96641cfcb0', 'rows': 1}, 24: {'sha256': '465fab847aafcd1e6ad8941529864876bbea7af37bf2e648eff268cac6ac0cdd', 'rows': 2}, 25: {'sha256': 'f242b75b402e389843eb787307a8ea3aaf0b898de6868b66131c97967af62361', 'rows': 3}, 26: {'sha256': 'bfe75f9a98c16a1264dad04a52b0e92604f254c3e3600ef4e9b47ff5e4ff54ef', 'rows': 2}, 27: {'sha256': 'ba4ca0c591a74862f52ce6a99ee900bddac5e77a2cc1bf207b36ced8ea377189', 'rows': 2}, 28: {'sha256': '995d15709a052c299948e41d8b3a652b926942e84b71e63ea81035884cb81855', 'rows': 3}, 29: {'sha256': 'aae6eb02682090392ce1c36c5d80ce3f76ee75be3a85c43ec59a599a3d4b9a90', 'rows': 3}, 30: {'sha256': 'ef0896e0dbb5e562c1eca92c395e102406b0a5ba1fb04357763bb28eacc86302', 'rows': 2}}

def check_sql(query, question_number):
    if not query or not query.strip():
        print("Write your SQL query first.")
        return
    try:
        result = pd.read_sql_query(query, conn)
    except Exception as e:
        print("SQL error:", e)
        return
    display(result)
    expected = EXPECTED_SQL_SIGNATURES[question_number]
    actual_hash, actual_rows = _signature(result, order_sensitive=(question_number == 15))
    if actual_hash == expected["sha256"]:
        print(f"✅ Q{question_number}: Correct! Your result matches the expected output ({actual_rows} rows).")
    else:
        print("❌ Not matching the expected output yet. Check your selected columns/order, JOIN conditions, filters, aggregation, and duplicates.")
        print(f"Your query returned {actual_rows} row(s). Re-read the question and try again.")


## 3. Practice questions

**Tip:** Each answer cell starts empty. Replace the empty string with your SQL query, keeping the triple quotes. Then run the cell.

### Q1. Transactions — Filter by type and amount

Display the transaction ID, transaction amount, and transaction type for transactions whose type is `Debit` and whose amount is greater than 10,000 but less than 50,000.

**Tables:** `transactions`  
**Expected output columns:** TRANSACTION_ID, AMOUNT, TRANSACTION_TYPE

In [68]:
# Run this cell to inspect the tables for Q1 before writing your SQL.
show_question_tables(1, ['transactions'], [])

QUESTION 1 — TABLE PREVIEW

No JOIN needed for this question.

============== TABLE: transactions ==============


,column,data type,primary key?
0,transaction_id,INTEGER,1
1,amount,REAL,0
2,transaction_type,TEXT,0


Rows: 5


,transaction_id,amount,transaction_type
0,1001,15000.0,Debit
1,1002,52000.0,Debit
2,1003,24000.0,Credit
3,1004,42000.0,Debit
4,1005,9000.0,Debit


In [69]:
query = """
-- Write your SQL query for Q1 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q1 here":
    check_sql(query, 1)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q2. Customers — Account type begins with S

Display the first name, contact, and balance of customers whose account type starts with the letter `S`.

**Tables:** `customers, accounts`  
**Expected output columns:** FIRST_NAME, CONTACT, BALANCE

In [70]:
# Run this cell to inspect the tables for Q2 before writing your SQL.
show_question_tables(2, ['customers', 'accounts'], ['customers.customer_id = accounts.customer_id'])

QUESTION 2 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • customers.customer_id = accounts.customer_id

============== TABLE: customers ==============


,column,data type,primary key?
0,customer_id,INTEGER,1
1,first_name,TEXT,0
2,last_name,TEXT,0
3,contact,TEXT,0


Rows: 5


,customer_id,first_name,last_name,contact
0,1,Aarav,Shah,9000000001
1,2,Meera,Rao,9000000002
2,3,Kabir,Das,9000000003
3,4,Isha,Patel,9000000004
4,5,Rohan,Singh,9000000005



============== TABLE: accounts ==============


,column,data type,primary key?
0,account_id,INTEGER,1
1,customer_id,INTEGER,0
2,account_type_id,INTEGER,0
3,account_type,TEXT,0
4,account_balance,REAL,0


Rows: 6


,account_id,customer_id,account_type_id,account_type,account_balance
0,501,1,10,Savings,75000.0
1,502,2,10,Savings Plus,45000.0
2,503,3,20,Current,125000.0
3,504,4,30,Salary,68000.0
4,505,5,20,Current,25000.0
5,506,1,20,Current,55000.0


In [71]:
query = """
-- Write your SQL query for Q2 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q2 here":
    check_sql(query, 2)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q3. Employee payroll — Basic and net pay

Display employee ID, employee name, basic salary, and net pay for employees whose basic salary is below 50,000.

**Tables:** `employee_payroll`  
**Expected output columns:** EMPID, EMPNAME, BASIC, NETPAY

In [72]:
# Run this cell to inspect the tables for Q3 before writing your SQL.
show_question_tables(3, ['employee_payroll'], [])

QUESTION 3 — TABLE PREVIEW

No JOIN needed for this question.

============== TABLE: employee_payroll ==============


,column,data type,primary key?
0,empid,INTEGER,1
1,empname,TEXT,0
2,basic,REAL,0
3,netpay,REAL,0


Rows: 4


,empid,empname,basic,netpay
0,11,Ananya Verma,42000.0,51000.0
1,12,Dev Kumar,55000.0,67000.0
2,13,Sara Khan,38500.0,46000.0
3,14,Arjun Mehta,62000.0,74000.0


In [73]:
query = """
-- Write your SQL query for Q3 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q3 here":
    check_sql(query, 3)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q4. Employees — Experience and joining date

Display the employee ID and employee name for employees who have more than 5 years of experience and joined after 1 January 2001.

**Tables:** `employee_info`  
**Expected output columns:** Employee ID, Employee Name

In [74]:
# Run this cell to inspect the tables for Q4 before writing your SQL.
show_question_tables(4, ['employee_info'], [])

QUESTION 4 — TABLE PREVIEW

No JOIN needed for this question.

============== TABLE: employee_info ==============


,column,data type,primary key?
0,empid,INTEGER,1
1,empname,TEXT,0
2,experience_years,REAL,0
3,joining_date,TEXT,0
4,deptid,INTEGER,0
5,employee_category,TEXT,0


Rows: 5


,empid,empname,experience_years,joining_date,deptid,employee_category
0,201,Ananya Verma,7.0,2018-06-15,1,A
1,202,Dev Kumar,4.0,2022-07-01,2,B
2,203,Sara Khan,9.0,2015-03-20,3,A
3,204,Arjun Mehta,6.0,2019-11-10,2,C
4,205,Meera Rao,2.0,2024-01-15,1,B


In [75]:
query = """
-- Write your SQL query for Q4 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q4 here":
    check_sql(query, 4)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q5. Courses — Schedule details

Display the course ID, course name, scheduled day, and start time for each course schedule.

**Tables:** `courses, course_schedule`  
**Expected output columns:** Course ID, Course Name, Day, Start Time

In [76]:
# Run this cell to inspect the tables for Q5 before writing your SQL.
show_question_tables(5, ['courses', 'course_schedule'], ['courses.course_id = course_schedule.course_id'])

QUESTION 5 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • courses.course_id = course_schedule.course_id

============== TABLE: courses ==============


,column,data type,primary key?
0,course_id,INTEGER,1
1,course_name,TEXT,0
2,course_no,TEXT,0


Rows: 4


,course_id,course_name,course_no
0,1,Database Systems,CS201
1,2,Data Analytics,CS305
2,3,Thermodynamics,ME202
3,4,Business Statistics,MG210



============== TABLE: course_schedule ==============


,column,data type,primary key?
0,schedule_id,INTEGER,1
1,course_id,INTEGER,0
2,day,TEXT,0
3,start_time,TEXT,0
4,faculty_name,TEXT,0


Rows: 4


,schedule_id,course_id,day,start_time,faculty_name
0,1,1,Monday,09:00,Dr Rao
1,2,2,Tuesday,11:00,Dr Sen
2,3,3,Wednesday,10:00,Dr Rao
3,4,4,Friday,14:00,Dr Iyer


In [77]:
query = """
-- Write your SQL query for Q5 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q5 here":
    check_sql(query, 5)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q6. Books — Date and category filter

Display the title, price, and ISBN of books published after 1 January 1980 that belong to the `Classic` category.

**Tables:** `books`  
**Expected output columns:** Title, Price, ISBN

In [78]:
# Run this cell to inspect the tables for Q6 before writing your SQL.
show_question_tables(6, ['books'], [])

QUESTION 6 — TABLE PREVIEW

No JOIN needed for this question.

============== TABLE: books ==============


,column,data type,primary key?
0,book_id,INTEGER,1
1,title,TEXT,0
2,price,REAL,0
3,isbn,TEXT,0
4,publish_date,TEXT,0
5,category,TEXT,0


Rows: 4


,book_id,title,price,isbn,publish_date,category
0,1,Classic Tales,450.0,9780000000011,1995-05-12,Classic
1,2,Old Poems,250.0,9780000000028,1975-02-01,Classic
2,3,Modern SQL,700.0,9780000000035,2021-09-20,Technology
3,4,The Odyssey,525.0,9780000000042,2000-01-01,Classic


In [79]:
query = """
-- Write your SQL query for Q6 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q6 here":
    check_sql(query, 6)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q7. Channel categories — Name contains M

Display the category ID and category name for channel categories whose name contains the letter `M`.

**Tables:** `channelcategory`  
**Expected output columns:** CATEGORYID, CATEGORYNAME

In [80]:
# Run this cell to inspect the tables for Q7 before writing your SQL.
show_question_tables(7, ['channelcategory'], [])

QUESTION 7 — TABLE PREVIEW

No JOIN needed for this question.

============== TABLE: channelcategory ==============


,column,data type,primary key?
0,categoryid,INTEGER,1
1,categoryname,TEXT,0


Rows: 5


,categoryid,categoryname
0,1,Movies
1,2,Music
2,3,Sports
3,4,News
4,5,Family


In [81]:
query = """
-- Write your SQL query for Q7 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q7 here":
    check_sql(query, 7)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q8. Trains — Name and destination station

Display the train ID and train name for trains whose name contains `M` and whose destination station is `PUNE`.

**Tables:** `train_details_tbl, train_station_tbl`  
**Expected output columns:** train_id, train_name

In [82]:
# Run this cell to inspect the tables for Q8 before writing your SQL.
show_question_tables(8, ['train_details_tbl', 'train_station_tbl'], ['train_details_tbl.train_to = train_station_tbl.station_id'])

QUESTION 8 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • train_details_tbl.train_to = train_station_tbl.station_id

============== TABLE: train_details_tbl ==============


,column,data type,primary key?
0,train_id,INTEGER,1
1,train_name,TEXT,0
2,train_type,TEXT,0
3,train_to,INTEGER,0
4,train_speed,REAL,0


Rows: 5


,train_id,train_name,train_type,train_to,train_speed
0,101,Mumbai Express,Express,1,75.0
1,102,Pune Mail,Mail,1,45.0
2,103,Metro Local,Passenger,2,35.0
3,104,Delhi Superfast,Superfast,3,110.0
4,105,Morning Intercity,Intercity,4,48.0



============== TABLE: train_station_tbl ==============


,column,data type,primary key?
0,station_id,INTEGER,1
1,station_name,TEXT,0


Rows: 4


,station_id,station_name
0,1,PUNE
1,2,MUMBAI
2,3,DELHI
3,4,CHENNAI


In [83]:
query = """
-- Write your SQL query for Q8 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q8 here":
    check_sql(query, 8)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q9. Leave information — Filter leave types

Display employee ID, leave type, and total leaves for employees with more than 10 total leaves and leave type `CL` or `ML`.

**Tables:** `leave_info`  
**Expected output columns:** EMPID, LEAVE_TYPE, TOTAL_LEAVES

In [84]:
# Run this cell to inspect the tables for Q9 before writing your SQL.
show_question_tables(9, ['leave_info'], [])

QUESTION 9 — TABLE PREVIEW

No JOIN needed for this question.

============== TABLE: leave_info ==============


,column,data type,primary key?
0,empid,INTEGER,0
1,leave_type,TEXT,0
2,total_leaves,INTEGER,0


Rows: 5


,empid,leave_type,total_leaves
0,11,CL,12
1,12,ML,14
2,13,EL,18
3,14,CL,8
4,15,ML,22


In [85]:
query = """
-- Write your SQL query for Q9 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q9 here":
    check_sql(query, 9)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q10. Employees — IT department

Display employee ID, employee name, department name, and basic salary for employees in the `IT` department.

**Tables:** `employee_info, department_info, salary_info`  
**Expected output columns:** EMPID, EMPNAME, DEPTNAME, BASIC

In [86]:
# Run this cell to inspect the tables for Q10 before writing your SQL.
show_question_tables(10, ['employee_info', 'department_info', 'salary_info'], ['employee_info.deptid = department_info.deptid', 'employee_info.employee_category = salary_info.employee_category'])

QUESTION 10 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • employee_info.deptid = department_info.deptid
  • employee_info.employee_category = salary_info.employee_category

============== TABLE: employee_info ==============


,column,data type,primary key?
0,empid,INTEGER,1
1,empname,TEXT,0
2,experience_years,REAL,0
3,joining_date,TEXT,0
4,deptid,INTEGER,0
5,employee_category,TEXT,0


Rows: 5


,empid,empname,experience_years,joining_date,deptid,employee_category
0,201,Ananya Verma,7.0,2018-06-15,1,A
1,202,Dev Kumar,4.0,2022-07-01,2,B
2,203,Sara Khan,9.0,2015-03-20,3,A
3,204,Arjun Mehta,6.0,2019-11-10,2,C
4,205,Meera Rao,2.0,2024-01-15,1,B



============== TABLE: department_info ==============


...

,deptid,deptname,location
0,1,IT,BANGALORE
1,2,Analytics,MUMBAI
2,3,HR,BANGALORE



============== TABLE: salary_info ==============


,column,data type,primary key?
0,employee_category,TEXT,1
1,basic,REAL,0
2,special_incentive_allowance,REAL,0


Rows: 3


,employee_category,basic,special_incentive_allowance
0,A,85000.0,12000.0
1,B,62000.0,8000.0
2,C,95000.0,15000.0


In [87]:
query = """
-- Write your SQL query for Q10 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q10 here":
    check_sql(query, 10)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q11. Employees — Bangalore allowance

Display employee ID, employee name, department name, and special incentive allowance for employees in departments located in `BANGALORE`.

**Tables:** `employee_info, department_info, salary_info`  
**Expected output columns:** EMPID, EMPNAME, DEPTNAME, SPECIAL_INCENTIVE_ALLOWANCE

In [88]:
# Run this cell to inspect the tables for Q11 before writing your SQL.
show_question_tables(11, ['employee_info', 'department_info', 'salary_info'], ['employee_info.deptid = department_info.deptid', 'employee_info.employee_category = salary_info.employee_category'])

QUESTION 11 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • employee_info.deptid = department_info.deptid
  • employee_info.employee_category = salary_info.employee_category

============== TABLE: employee_info ==============


,column,data type,primary key?
0,empid,INTEGER,1
1,empname,TEXT,0
2,experience_years,REAL,0
3,joining_date,TEXT,0
4,deptid,INTEGER,0
5,employee_category,TEXT,0


Rows: 5


,empid,empname,experience_years,joining_date,deptid,employee_category
0,201,Ananya Verma,7.0,2018-06-15,1,A
1,202,Dev Kumar,4.0,2022-07-01,2,B
2,203,Sara Khan,9.0,2015-03-20,3,A
3,204,Arjun Mehta,6.0,2019-11-10,2,C
4,205,Meera Rao,2.0,2024-01-15,1,B



============== TABLE: department_info ==============


...

,deptid,deptname,location
0,1,IT,BANGALORE
1,2,Analytics,MUMBAI
2,3,HR,BANGALORE



============== TABLE: salary_info ==============


,column,data type,primary key?
0,employee_category,TEXT,1
1,basic,REAL,0
2,special_incentive_allowance,REAL,0


Rows: 3


,employee_category,basic,special_incentive_allowance
0,A,85000.0,12000.0
1,B,62000.0,8000.0
2,C,95000.0,15000.0


In [89]:
query = """
-- Write your SQL query for Q11 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q11 here":
    check_sql(query, 11)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q12. Accounts — Average balance by account type

For each account type ID, display the average account balance. Show only groups whose average balance is at least 50,000.

**Tables:** `accounts`  
**Expected output columns:** Account_Type_ID, Average

In [90]:
# Run this cell to inspect the tables for Q12 before writing your SQL.
show_question_tables(12, ['accounts'], [])

QUESTION 12 — TABLE PREVIEW

No JOIN needed for this question.

============== TABLE: accounts ==============


,column,data type,primary key?
0,account_id,INTEGER,1
1,customer_id,INTEGER,0
2,account_type_id,INTEGER,0
3,account_type,TEXT,0
4,account_balance,REAL,0


Rows: 6


,account_id,customer_id,account_type_id,account_type,account_balance
0,501,1,10,Savings,75000.0
1,502,2,10,Savings Plus,45000.0
2,503,3,20,Current,125000.0
3,504,4,30,Salary,68000.0
4,505,5,20,Current,25000.0
5,506,1,20,Current,55000.0


In [91]:
query = """
-- Write your SQL query for Q12 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q12 here":
    check_sql(query, 12)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q13. Customers — Accounts with high balance

Display the first name, last name, and account ID of customers whose account balance is greater than 50,000.

**Tables:** `customers, accounts`  
**Expected output columns:** FIRST_NAME, LAST_NAME, ACCOUNT_ID

In [92]:
# Run this cell to inspect the tables for Q13 before writing your SQL.
show_question_tables(13, ['customers', 'accounts'], ['customers.customer_id = accounts.customer_id'])

QUESTION 13 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • customers.customer_id = accounts.customer_id

============== TABLE: customers ==============


,column,data type,primary key?
0,customer_id,INTEGER,1
1,first_name,TEXT,0
2,last_name,TEXT,0
3,contact,TEXT,0


Rows: 5


,customer_id,first_name,last_name,contact
0,1,Aarav,Shah,9000000001
1,2,Meera,Rao,9000000002
2,3,Kabir,Das,9000000003
3,4,Isha,Patel,9000000004
4,5,Rohan,Singh,9000000005



============== TABLE: accounts ==============


,column,data type,primary key?
0,account_id,INTEGER,1
1,customer_id,INTEGER,0
2,account_type_id,INTEGER,0
3,account_type,TEXT,0
4,account_balance,REAL,0


Rows: 6


,account_id,customer_id,account_type_id,account_type,account_balance
0,501,1,10,Savings,75000.0
1,502,2,10,Savings Plus,45000.0
2,503,3,20,Current,125000.0
3,504,4,30,Salary,68000.0
4,505,5,20,Current,25000.0
5,506,1,20,Current,55000.0


In [93]:
query = """
-- Write your SQL query for Q13 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q13 here":
    check_sql(query, 13)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q14. Staff — Salary threshold

Display the staff member's first name, position, and salary for staff whose salary is greater than 90,000.

**Tables:** `staff`  
**Expected output columns:** STAFF FIRST NAME, POSITION, SALARY

In [94]:
# Run this cell to inspect the tables for Q14 before writing your SQL.
show_question_tables(14, ['staff'], [])

QUESTION 14 — TABLE PREVIEW

No JOIN needed for this question.

============== TABLE: staff ==============


,column,data type,primary key?
0,staff_id,INTEGER,1
1,firstname,TEXT,0
2,position,TEXT,0
3,salary,REAL,0


Rows: 4


,staff_id,firstname,position,salary
0,1,Nisha,Analyst,95000.0
1,2,Vikram,Manager,125000.0
2,3,Fatima,Associate,72000.0
3,4,Karan,Consultant,102000.0


In [95]:
query = """
-- Write your SQL query for Q14 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q14 here":
    check_sql(query, 14)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q15. Hospital — Unpaid patient billing

Display each patient's first name, last name, email, admission date, and total billing amount for patients whose billing payment status is `Unpaid`. Sort by total billing amount from highest to lowest.

**Tables:** `patient, billing`  
**Expected output columns:** Patient name fields, patient email, admission date, total billing

In [96]:
# Run this cell to inspect the tables for Q15 before writing your SQL.
show_question_tables(15, ['patient', 'billing'], ['patient.patient_id = billing.patient_id'])

QUESTION 15 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • patient.patient_id = billing.patient_id

============== TABLE: patient ==============


,column,data type,primary key?
0,patient_id,INTEGER,1
1,first_name,TEXT,0
2,last_name,TEXT,0
3,email,TEXT,0
4,admissiondate,TEXT,0


Rows: 3


,patient_id,first_name,last_name,email,admissiondate
0,1,Aditi,Sharma,aditi@example.com,2024-01-10
1,2,Rahul,Jain,rahul@example.com,2024-02-05
2,3,Neha,Kapoor,neha@example.com,2024-03-11



============== TABLE: billing ==============


,column,data type,primary key?
0,billing_id,INTEGER,1
1,patient_id,INTEGER,0
2,totalamount,REAL,0
3,paymentstatus,TEXT,0


Rows: 4


,billing_id,patient_id,totalamount,paymentstatus
0,101,1,25000.0,Unpaid
1,102,1,7000.0,Unpaid
2,103,2,18000.0,Paid
3,104,3,46000.0,Unpaid


In [97]:
query = """
-- Write your SQL query for Q15 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q15 here":
    check_sql(query, 15)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q16. Flights — Singapore Airlines departures

Display the flight ID, departure date, and departure time for flights operated by `Singapore Airlines`.

**Tables:** `airline, airplane, flight`  
**Expected output columns:** FLIGHT_ID, Departure_date, Departure_Time

In [98]:
# Run this cell to inspect the tables for Q16 before writing your SQL.
show_question_tables(16, ['airline', 'airplane', 'flight'], ['airline.airline_id = airplane.airline_id', 'airplane.airplane_id = flight.airplane_id'])

QUESTION 16 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • airline.airline_id = airplane.airline_id
  • airplane.airplane_id = flight.airplane_id

============== TABLE: airline ==============


,column,data type,primary key?
0,airline_id,INTEGER,1
1,name,TEXT,0
2,country,TEXT,0


Rows: 3


,airline_id,name,country
0,1,Singapore Airlines,Singapore
1,2,Air India,India
2,3,Lufthansa,Germany



============== TABLE: airplane ==============


...

,airplane_id,airline_id,modelnumber,manufacturer
0,1,1,A350-900,Airbus
1,2,2,B787-8,Boeing
2,3,3,A320neo,Airbus
3,4,2,A321,Airbus



============== TABLE: flight ==============


,column,data type,primary key?
0,flight_id,TEXT,1
1,airplane_id,INTEGER,0
2,departure_date,TEXT,0
3,departure_time,TEXT,0
4,arrival_date,TEXT,0
5,flight_from,TEXT,0
6,flight_to,TEXT,0


Rows: 5


,flight_id,airplane_id,departure_date,departure_time,arrival_date,flight_from,flight_to
0,F101,1,2024-02-10,08:30,2024-02-11,Hong Kong,Paris
1,F211,2,2024-02-10,11:15,2024-02-11,Hong Kong,Paris
2,F304,3,2024-02-11,14:00,2024-02-11,Delhi,Frankfurt
3,F411,4,2024-02-11,19:45,2024-02-11,Hong Kong,Singapore
4,4,2,2024-02-11,21:00,2024-02-12,Hong Kong,Singapore


In [99]:
query = """
-- Write your SQL query for Q16 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q16 here":
    check_sql(query, 16)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q17. Airplanes — Airbus models

Display the airplane ID and model number for airplanes manufactured by `Airbus`.

**Tables:** `airplane`  
**Expected output columns:** AIRPLANE_ID, MODELNUMBER

In [100]:
# Run this cell to inspect the tables for Q17 before writing your SQL.
show_question_tables(17, ['airplane'], [])

QUESTION 17 — TABLE PREVIEW

No JOIN needed for this question.

============== TABLE: airplane ==============


,column,data type,primary key?
0,airplane_id,INTEGER,1
1,airline_id,INTEGER,0
2,modelnumber,TEXT,0
3,manufacturer,TEXT,0


Rows: 4


,airplane_id,airline_id,modelnumber,manufacturer
0,1,1,A350-900,Airbus
1,2,2,B787-8,Boeing
2,3,3,A320neo,Airbus
3,4,2,A321,Airbus


In [101]:
query = """
-- Write your SQL query for Q17 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q17 here":
    check_sql(query, 17)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q18. Students — Registrations in 2012

Display the last names of students who registered during the year 2012.

**Tables:** `student, registration`  
**Expected output columns:** LAST_NAME

In [102]:
# Run this cell to inspect the tables for Q18 before writing your SQL.
show_question_tables(18, ['student', 'registration'], ['student.student_id = registration.student_id'])

QUESTION 18 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • student.student_id = registration.student_id

============== TABLE: student ==============


,column,data type,primary key?
0,student_id,INTEGER,1
1,first_name,TEXT,0
2,last_name,TEXT,0


Rows: 4


,student_id,first_name,last_name
0,1,Asha,Menon
1,2,Vivek,Nair
2,3,Riya,Bose
3,4,Omar,Ali



============== TABLE: registration ==============


,column,data type,primary key?
0,registration_id,INTEGER,1
1,student_id,INTEGER,0
2,reg_date,TEXT,0


Rows: 4


,registration_id,student_id,reg_date
0,1,1,2012-03-14
1,2,2,2014-08-10
2,3,3,2012-11-02
3,4,4,2011-01-20


In [103]:
query = """
-- Write your SQL query for Q18 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q18 here":
    check_sql(query, 18)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q19. Cabin crew — Name and flight filters

Display cabin crew ID, first name, last name, contact, and flight ID for crew members whose first name starts with `A` and whose flight ID ends with `1`.

**Tables:** `cabincrew, flight`  
**Expected output columns:** CABINCREW_ID, FIRST_NAME, LAST_NAME, CONTACT, FLIGHT_ID

In [104]:
# Run this cell to inspect the tables for Q19 before writing your SQL.
show_question_tables(19, ['cabincrew', 'flight'], ['cabincrew.flight_id = flight.flight_id'])

QUESTION 19 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • cabincrew.flight_id = flight.flight_id

============== TABLE: cabincrew ==============


,column,data type,primary key?
0,cabincrew_id,INTEGER,1
1,first_name,TEXT,0
2,last_name,TEXT,0
3,contact,TEXT,0
4,flight_id,TEXT,0


Rows: 4


,cabincrew_id,first_name,last_name,contact,flight_id
0,1,Anika,Sen,8000000001,F101
1,2,Aman,Rao,8000000002,F211
2,3,Rita,Das,8000000003,F304
3,4,Arun,Jain,8000000004,F411



============== TABLE: flight ==============


,column,data type,primary key?
0,flight_id,TEXT,1
1,airplane_id,INTEGER,0
2,departure_date,TEXT,0
3,departure_time,TEXT,0
4,arrival_date,TEXT,0
5,flight_from,TEXT,0
6,flight_to,TEXT,0


Rows: 5


,flight_id,airplane_id,departure_date,departure_time,arrival_date,flight_from,flight_to
0,F101,1,2024-02-10,08:30,2024-02-11,Hong Kong,Paris
1,F211,2,2024-02-10,11:15,2024-02-11,Hong Kong,Paris
2,F304,3,2024-02-11,14:00,2024-02-11,Delhi,Frankfurt
3,F411,4,2024-02-11,19:45,2024-02-11,Hong Kong,Singapore
4,4,2,2024-02-11,21:00,2024-02-12,Hong Kong,Singapore


In [105]:
query = """
-- Write your SQL query for Q19 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q19 here":
    check_sql(query, 19)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q20. Flights — Passenger and baggage totals

For flights arriving in `Paris` on `2024-02-11`, display flight ID, total passengers, and total baggage.

**Tables:** `flight, boardingpass`  
**Expected output columns:** Flight_ID, Total_Passengers, Total_Baggage

In [106]:
# Run this cell to inspect the tables for Q20 before writing your SQL.
show_question_tables(20, ['flight', 'boardingpass'], ['flight.flight_id = boardingpass.flight_id'])

QUESTION 20 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • flight.flight_id = boardingpass.flight_id

============== TABLE: flight ==============


,column,data type,primary key?
0,flight_id,TEXT,1
1,airplane_id,INTEGER,0
2,departure_date,TEXT,0
3,departure_time,TEXT,0
4,arrival_date,TEXT,0
5,flight_from,TEXT,0
6,flight_to,TEXT,0


Rows: 5


,flight_id,airplane_id,departure_date,departure_time,arrival_date,flight_from,flight_to
0,F101,1,2024-02-10,08:30,2024-02-11,Hong Kong,Paris
1,F211,2,2024-02-10,11:15,2024-02-11,Hong Kong,Paris
2,F304,3,2024-02-11,14:00,2024-02-11,Delhi,Frankfurt
3,F411,4,2024-02-11,19:45,2024-02-11,Hong Kong,Singapore
4,4,2,2024-02-11,21:00,2024-02-12,Hong Kong,Singapore



============== TABLE: boardingpass ==============


,column,data type,primary key?
0,boardingpass_id,INTEGER,1
1,passenger_id,INTEGER,0
2,flight_id,TEXT,0
3,baggage,REAL,0
4,meal,TEXT,0


Rows: 4


,boardingpass_id,passenger_id,flight_id,baggage,meal
0,1,1,F101,22.0,Vegetarian
1,2,2,F101,18.0,Non-Vegetarian
2,3,3,F211,25.0,Vegetarian
3,4,4,4,30.0,Vegetarian


In [107]:
query = """
-- Write your SQL query for Q20 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q20 here":
    check_sql(query, 20)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q21. Products — Count in Women category

Display the number of products associated with the `Women` category.

**Tables:** `product, product_category, category`  
**Expected output columns:** product_count

In [108]:
# Run this cell to inspect the tables for Q21 before writing your SQL.
show_question_tables(21, ['product', 'product_category', 'category'], ['product.product_id = product_category.product_id', 'product_category.category_id = category.category_id'])

QUESTION 21 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • product.product_id = product_category.product_id
  • product_category.category_id = category.category_id

============== TABLE: product ==============


,column,data type,primary key?
0,product_id,INTEGER,1
1,name,TEXT,0


Rows: 4


,product_id,name
0,1,Blue Dress
1,2,Running Shoes
2,3,Desk Lamp
3,4,Travel Bag



============== TABLE: product_category ==============


...

,product_id,category_id
0,1,1
1,2,2
2,3,3
3,4,4
4,2,1



============== TABLE: category ==============


,column,data type,primary key?
0,category_id,INTEGER,1
1,name,TEXT,0


Rows: 4


,category_id,name
0,1,Women
1,2,Sports
2,3,Home
3,4,Travel


In [109]:
query = """
-- Write your SQL query for Q21 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q21 here":
    check_sql(query, 21)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q22. Trains — Speed under 50

Display train name and train type for trains whose speed is less than 50.

**Tables:** `train_details_tbl`  
**Expected output columns:** TRAIN_NAME, TRAIN_TYPE

In [110]:
# Run this cell to inspect the tables for Q22 before writing your SQL.
show_question_tables(22, ['train_details_tbl'], [])

QUESTION 22 — TABLE PREVIEW

No JOIN needed for this question.

============== TABLE: train_details_tbl ==============


,column,data type,primary key?
0,train_id,INTEGER,1
1,train_name,TEXT,0
2,train_type,TEXT,0
3,train_to,INTEGER,0
4,train_speed,REAL,0


Rows: 5


,train_id,train_name,train_type,train_to,train_speed
0,101,Mumbai Express,Express,1,75.0
1,102,Pune Mail,Mail,1,45.0
2,103,Metro Local,Passenger,2,35.0
3,104,Delhi Superfast,Superfast,3,110.0
4,105,Morning Intercity,Intercity,4,48.0


In [111]:
query = """
-- Write your SQL query for Q22 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q22 here":
    check_sql(query, 22)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q23. Passengers — Vegetarian meal on a flight

Display the distinct first names and contact details of passengers on flight ID `4` departing from `Hong Kong` who selected a `Vegetarian` meal.

**Tables:** `passenger, boardingpass, flight`  
**Expected output columns:** FIRST_NAME, CONTACT

In [112]:
# Run this cell to inspect the tables for Q23 before writing your SQL.
show_question_tables(23, ['passenger', 'boardingpass', 'flight'], ['passenger.passenger_id = boardingpass.passenger_id', 'boardingpass.flight_id = flight.flight_id'])

QUESTION 23 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • passenger.passenger_id = boardingpass.passenger_id
  • boardingpass.flight_id = flight.flight_id

============== TABLE: passenger ==============


,column,data type,primary key?
0,passenger_id,INTEGER,1
1,first_name,TEXT,0
2,contact,TEXT,0


Rows: 4


,passenger_id,first_name,contact
0,1,Maya,7000000001
1,2,Rohan,7000000002
2,3,Ira,7000000003
3,4,Kunal,7000000004



============== TABLE: boardingpass ==============


...

,boardingpass_id,passenger_id,flight_id,baggage,meal
0,1,1,F101,22.0,Vegetarian
1,2,2,F101,18.0,Non-Vegetarian
2,3,3,F211,25.0,Vegetarian
3,4,4,4,30.0,Vegetarian



============== TABLE: flight ==============


,column,data type,primary key?
0,flight_id,TEXT,1
1,airplane_id,INTEGER,0
2,departure_date,TEXT,0
3,departure_time,TEXT,0
4,arrival_date,TEXT,0
5,flight_from,TEXT,0
6,flight_to,TEXT,0


Rows: 5


,flight_id,airplane_id,departure_date,departure_time,arrival_date,flight_from,flight_to
0,F101,1,2024-02-10,08:30,2024-02-11,Hong Kong,Paris
1,F211,2,2024-02-10,11:15,2024-02-11,Hong Kong,Paris
2,F304,3,2024-02-11,14:00,2024-02-11,Delhi,Frankfurt
3,F411,4,2024-02-11,19:45,2024-02-11,Hong Kong,Singapore
4,4,2,2024-02-11,21:00,2024-02-12,Hong Kong,Singapore


In [113]:
query = """
-- Write your SQL query for Q23 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q23 here":
    check_sql(query, 23)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q24. Products — Orders in transit hub

Display distinct product ID and product name for products whose order delivery status is `In the transit hub`.

**Tables:** `product, order_item, order_delivery`  
**Expected output columns:** PRODUCT_ID, NAME

In [114]:
# Run this cell to inspect the tables for Q24 before writing your SQL.
show_question_tables(24, ['product', 'order_item', 'order_delivery'], ['product.product_id = order_item.product_id', 'order_item.order_delivery_id = order_delivery.order_delivery_id'])

QUESTION 24 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • product.product_id = order_item.product_id
  • order_item.order_delivery_id = order_delivery.order_delivery_id

============== TABLE: product ==============


,column,data type,primary key?
0,product_id,INTEGER,1
1,name,TEXT,0


Rows: 4


,product_id,name
0,1,Blue Dress
1,2,Running Shoes
2,3,Desk Lamp
3,4,Travel Bag



============== TABLE: order_item ==============


...

,order_item_id,product_id,order_delivery_id
0,1,1,501
1,2,2,502
2,3,3,501
3,4,4,503



============== TABLE: order_delivery ==============


,column,data type,primary key?
0,order_delivery_id,INTEGER,1
1,status,TEXT,0


Rows: 3


,order_delivery_id,status
0,501,In the transit hub
1,502,Delivered
2,503,Processing


In [115]:
query = """
-- Write your SQL query for Q24 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q24 here":
    check_sql(query, 24)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q25. Artists — Name contains a digit

Display artist ID and name for artists whose name contains at least one digit.

**Tables:** `artist`  
**Expected output columns:** ARTIST_ID, NAME

In [116]:
# Run this cell to inspect the tables for Q25 before writing your SQL.
show_question_tables(25, ['artist'], [])

QUESTION 25 — TABLE PREVIEW

No JOIN needed for this question.

============== TABLE: artist ==============


,column,data type,primary key?
0,artist_id,INTEGER,1
1,name,TEXT,0


Rows: 5


,artist_id,name
0,1,Studio 54
1,2,Adele
2,3,Band 2
3,4,The Weeknd
4,5,Artist 007


In [117]:
query = """
-- Write your SQL query for Q25 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q25 here":
    check_sql(query, 25)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q26. Messages — Find Hello

Display message ID and content for messages whose content contains the word `Hello`.

**Tables:** `message`  
**Expected output columns:** MESSAGE_ID, CONTENT

In [118]:
# Run this cell to inspect the tables for Q26 before writing your SQL.
show_question_tables(26, ['message'], [])

QUESTION 26 — TABLE PREVIEW

No JOIN needed for this question.

============== TABLE: message ==============


,column,data type,primary key?
0,message_id,INTEGER,1
1,content,TEXT,0


Rows: 4


,message_id,content
0,1,Hello everyone
1,2,Good morning
2,3,Say hello to the team
3,4,Thanks for joining


In [119]:
query = """
-- Write your SQL query for Q26 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q26 here":
    check_sql(query, 26)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q27. Drivers — In-use vehicles with plate ending in 0

Display the driver's full name, license number, and plate number for drivers whose vehicle status is `In Use` and whose plate number ends with `0`.

**Tables:** `driver, vehicle`  
**Expected output columns:** Name, License_Number, Plate_Number

In [120]:
# Run this cell to inspect the tables for Q27 before writing your SQL.
show_question_tables(27, ['driver', 'vehicle'], ['driver.driver_id = vehicle.driver_id'])

QUESTION 27 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • driver.driver_id = vehicle.driver_id

============== TABLE: driver ==============


,column,data type,primary key?
0,driver_id,INTEGER,1
1,first_name,TEXT,0
2,last_name,TEXT,0
3,license_number,TEXT,0
4,rating,REAL,0


Rows: 4


,driver_id,first_name,last_name,license_number,rating
0,1,Aman,Gupta,DL1001,4.8
1,2,Nisha,Roy,DL1002,4.3
2,3,Karan,Das,DL1003,4.6
3,4,Meera,Shah,DL1004,4.9



============== TABLE: vehicle ==============


,column,data type,primary key?
0,vehicle_id,INTEGER,1
1,driver_id,INTEGER,0
2,plate_number,TEXT,0
3,status,TEXT,0


Rows: 4


,vehicle_id,driver_id,plate_number,status
0,1,1,UP32AB120,In Use
1,2,2,UP32CD321,In Use
2,3,3,UP32EF450,In Use
3,4,4,UP32GH770,Available


In [121]:
query = """
-- Write your SQL query for Q27 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q27 here":
    check_sql(query, 27)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q28. Drivers — High rating and non-cancelled booking

Display license number, vehicle ID, driver rating, and booking ID for drivers with a rating of at least 4.5 whose booking status is not `Cancelled`.

**Tables:** `driver, vehicle, booking`  
**Expected output columns:** License_Number, Vehicle_ID, Rating, Booking_ID

In [122]:
# Run this cell to inspect the tables for Q28 before writing your SQL.
show_question_tables(28, ['driver', 'vehicle', 'booking'], ['driver.driver_id = vehicle.driver_id', 'vehicle.vehicle_id = booking.vehicle_id'])

QUESTION 28 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • driver.driver_id = vehicle.driver_id
  • vehicle.vehicle_id = booking.vehicle_id

============== TABLE: driver ==============


,column,data type,primary key?
0,driver_id,INTEGER,1
1,first_name,TEXT,0
2,last_name,TEXT,0
3,license_number,TEXT,0
4,rating,REAL,0


Rows: 4


,driver_id,first_name,last_name,license_number,rating
0,1,Aman,Gupta,DL1001,4.8
1,2,Nisha,Roy,DL1002,4.3
2,3,Karan,Das,DL1003,4.6
3,4,Meera,Shah,DL1004,4.9



============== TABLE: vehicle ==============


...

,vehicle_id,driver_id,plate_number,status
0,1,1,UP32AB120,In Use
1,2,2,UP32CD321,In Use
2,3,3,UP32EF450,In Use
3,4,4,UP32GH770,Available



============== TABLE: booking ==============


,column,data type,primary key?
0,booking_id,INTEGER,1
1,vehicle_id,INTEGER,0
2,status,TEXT,0


Rows: 5


,booking_id,vehicle_id,status
0,701,1,Confirmed
1,702,1,Cancelled
2,703,2,Confirmed
3,704,3,Completed
4,705,4,Confirmed


In [123]:
query = """
-- Write your SQL query for Q28 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q28 here":
    check_sql(query, 28)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q29. Viewers — Count repeated names

Display each viewer name and the number of viewers having that name.

**Tables:** `viewer`  
**Expected output columns:** viewername, name_count

In [124]:
# Run this cell to inspect the tables for Q29 before writing your SQL.
show_question_tables(29, ['viewer'], [])

QUESTION 29 — TABLE PREVIEW

No JOIN needed for this question.

============== TABLE: viewer ==============


,column,data type,primary key?
0,viewer_id,INTEGER,1
1,viewername,TEXT,0


Rows: 6


,viewer_id,viewername
0,1,Aarav
1,2,Meera
2,3,Aarav
3,4,Riya
4,5,Meera
5,6,Aarav


In [125]:
query = """
-- Write your SQL query for Q29 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q29 here":
    check_sql(query, 29)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


### Q30. Users — Contacts with an Engineer job

Display the distinct full names of users who are contacts of users whose job title contains `Engineer`.

**Tables:** `users, contacts, jobs`  
**Expected output columns:** FULLNAME

In [126]:
# Run this cell to inspect the tables for Q30 before writing your SQL.
show_question_tables(30, ['users', 'contacts', 'jobs'], ['users AS u: u.user_id = contacts.user_id (the user whose contact list is being checked)', 'contacts.contact_id = users AS u2.user_id (the contact person)', 'users AS u2.user_id = jobs.user_id (the contact person’s job record)', 'Because users is used twice, give it two aliases: u and u2.'])

QUESTION 30 — TABLE PREVIEW

JOIN GUIDE (relationships only; write the SQL yourself):
  • users AS u: u.user_id = contacts.user_id (the user whose contact list is being checked)
  • contacts.contact_id = users AS u2.user_id (the contact person)
  • users AS u2.user_id = jobs.user_id (the contact person’s job record)
  • Because users is used twice, give it two aliases: u and u2.

============== TABLE: users ==============


,column,data type,primary key?
0,user_id,INTEGER,1
1,first_name,TEXT,0
2,last_name,TEXT,0


Rows: 5


,user_id,first_name,last_name
0,1,Anaya,Sharma
1,2,Vikram,Singh
2,3,Kavya,Rao
3,4,Arjun,Mehta
4,5,Neel,Kapoor



============== TABLE: contacts ==============


...

,user_id,contact_id
0,1,2
1,1,3
2,4,5
3,2,3



============== TABLE: jobs ==============


,column,data type,primary key?
0,job_id,INTEGER,1
1,user_id,INTEGER,0
2,job_title,TEXT,0


Rows: 4


,job_id,user_id,job_title
0,1,2,Software Engineer
1,2,3,Data Analyst
2,3,5,Mechanical Engineer
3,4,4,Product Manager


In [127]:
query = """
-- Write your SQL query for Q30 here

"""
if query.strip() and query.strip() != "-- Write your SQL query for Q30 here":
    check_sql(query, 30)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.


## Extra scratch cell

Use this whenever you want to test a query outside the 30 questions.

In [128]:
query = """
-- Write an extra SQL query here

"""
if query.strip() and query.strip() != "-- Write an extra SQL query here":
    run_sql(query)
else:
    print("Write your SQL query above, then run this cell.")

Write your SQL query above, then run this cell.
